# UDCF aplicado a Hillstrom - REPLICA do pipeline do usuario

Este notebook reproduz, celula por celula, exatamente as escolhas feitas no script `udcf_hillstrom_colab_corrigido.py` (o script que o usuario escreveu e rodou). Nenhuma escolha de modelagem foi alterada em relacao ao script original - e uma checagem de reprodutibilidade, para confirmar que o resultado bate com o que ja foi obtido antes.

Configuracao (identica ao script original do usuario):
- **Hiperparametros**: `min_node_size=5` (patch em `ForestTestUtilities.cpp`, default e 50), `alpha=0.01` (default 0.05), `imbalance_penalty=0.0` (default 0.01)
- **Codificacao de `zip_code`/`channel`**: ordinal (`zip_code_num`, `channel_num`, 0/1/2) -> 7 features
- **Amostra**: split 80/20 estratificado por `segment` (`random_state=42`), treino com 80%, CATE estimado nos 20% de teste via `predict` (nao `predict_oob`)
- **Zip usado**: `Code/Model/CF_DT/UDCF_RCT.zip` (mesmo caminho do script original; o codigo-fonte C++ e identico ao de `Code/Model/LBCF/LBCF_RCT.zip`, ja conferido linha a linha)

## 1. Clonar o repositorio e instalar ferramentas de build

In [ ]:
!git clone -q https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS.git
!apt-get -qq update && apt-get -qq install -y cmake g++

## 2. Extrair o codigo C++ do UDCF (pasta CF_DT, igual ao script original)

In [ ]:
import zipfile

BASE = "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/CF_DT"
with zipfile.ZipFile(f"{BASE}/UDCF_RCT.zip") as z:
    z.extractall(BASE)

## 3. Carregar a base Hillstrom, codificar (ordinal) e dividir treino/teste 80/20

Mesma codificacao e mesmo split do script original do usuario.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

HILLSTROM_URL = (
    "http://www.minethatdata.com/"
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)
df_h = pd.read_csv(HILLSTROM_URL)
print("Base Hillstrom original:", df_h.shape)

df_udcf = df_h.copy()
df_udcf["zip_code_num"] = df_udcf["zip_code"].map({"Urban": 0, "Surburban": 1, "Rural": 2})
df_udcf["channel_num"] = df_udcf["channel"].map({"Phone": 0, "Web": 1, "Multichannel": 2})

df_udcf["T_mens"] = (df_udcf["segment"] == "Mens E-Mail").astype(int)
df_udcf["T_womens"] = (df_udcf["segment"] == "Womens E-Mail").astype(int)

cols_udcf = [
    "recency", "history", "mens", "womens", "newbie",
    "zip_code_num", "channel_num",
    "conversion", "T_mens", "T_womens",
]
df_final = df_udcf[cols_udcf].copy()

if df_final.isna().any().any():
    raise ValueError("Valores ausentes encontrados apos a codificacao.")

train_idx, test_idx = train_test_split(
    df_h.index, test_size=0.20, random_state=42, stratify=df_h["segment"],
)

train_h = df_final.loc[train_idx].copy()
test_h = df_final.loc[test_idx].copy()

TRAIN_PATH = f"{BASE}/UDCF_RCT/core/train_hillstrom_udcf.csv"
TEST_PATH = f"{BASE}/UDCF_RCT/core/test_hillstrom_udcf.csv"
TEST_INDEX_PATH = f"{BASE}/UDCF_RCT/core/test_hillstrom_indices.csv"

train_h.to_csv(TRAIN_PATH, index=False, header=False, sep=" ")
test_h.to_csv(TEST_PATH, index=False, header=False, sep=" ")
pd.DataFrame({"original_index": test_idx}).to_csv(TEST_INDEX_PATH, index=False)

print("Formato adaptado ao UDCF:", df_final.shape)
print("Treino:", train_h.shape, "| Teste:", test_h.shape)

## 4. Ajustar os hiperparametros da floresta (patch em `ForestTestUtilities.cpp`)

Mesmas 3 alteracoes do script original do usuario.

In [ ]:
OPTIONS_PATH = f"{BASE}/UDCF_RCT/core/src/utilities/ForestTestUtilities.cpp"

with open(OPTIONS_PATH, encoding="utf-8") as f:
    options_code = f.read()

replacements = {
    "uint min_node_size = 50;": "uint min_node_size = 5;",
    "double alpha = 0.05;": "double alpha = 0.01;",
    "double imbalance_penalty = 0.01;": "double imbalance_penalty = 0.0;",
}

for old, new in replacements.items():
    if old not in options_code:
        raise RuntimeError(f"Nao encontrei no codigo-fonte a configuracao esperada:\n{old}")
    options_code = options_code.replace(old, new, 1)

with open(OPTIONS_PATH, "w", encoding="utf-8") as f:
    f.write(options_code)

print("Hiperparametros ajustados: min_node_size=5, alpha=0.01, imbalance_penalty=0.0")

## 5. Adaptar `main.cpp` (mesma logica do script original: treino/teste separados, `predict` nao-OOB)

In [ ]:
main_cpp = r'''
#include <iostream>
#include <string>
#include <unistd.h>

#include "tree/Tree.h"
#include "prediction/DefaultPredictionStrategy.h"
#include "commons/utility.h"
#include "forest/ForestPredictor.h"
#include "forest/ForestTrainer.h"
#include "utilities/FileTestUtilities.h"
#include "utilities/ForestTestUtilities.h"
#include "forest/ForestTrainers.h"
#include "forest/ForestPredictors.h"

using namespace grf;

void update_predictions_file(
    const std::string& file_name,
    const std::vector<Prediction>& predictions) {

  std::vector<std::vector<double>> values;
  values.reserve(predictions.size());

  for (const auto& prediction : predictions) {
    values.push_back(prediction.get_predictions());
  }

  FileTestUtilities::write_csv_file(file_name, values);
  std::cout << "success! predictions dump to "
            << file_name << std::endl;
}

int main() {

    char tmp[256];
    getcwd(tmp, 256);
    std::cout << "Current working directory: "
              << tmp << std::endl;

    auto data_vec = load_data("../train_hillstrom_udcf.csv");

    Data data(data_vec);
    data.set_outcome_index(7);
    data.set_treatment_index({8, 9});

    auto data_vec2 = load_data("../test_hillstrom_udcf.csv");

    Data data2(data_vec2);
    data2.set_outcome_index(7);
    data2.set_treatment_index({8});

    size_t num_treatments = 2;

    ForestTrainer trainer =
        udcf_trainer(num_treatments, 1, true);

    ForestOptions options =
        ForestTestUtilities::default_options(true, 1);

    Forest forest = trainer.train(data, options);

    std::cout << "Numero de arvores = "
              << forest.get_trees().size()
              << std::endl;

    ForestPredictor predictor =
        udcf_predictor(1, num_treatments, 1);

    std::vector<Prediction> predictions =
        predictor.predict(forest, data, data2, false);

    update_predictions_file(
        "../hillstrom_predictions.txt",
        predictions
    );

    return 0;
}
'''

with open(f"{BASE}/UDCF_RCT/core/main.cpp", "w") as f:
    f.write(main_cpp)

print("main.cpp gerado com sucesso.")

## 6. Compilar e executar

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/CF_DT/UDCF_RCT/core
rm -rf build
mkdir build
cd build
cmake ..
make -j2
./UDCF_RCT

## 7. Ler e verificar os CATEs

In [ ]:
df_cate = pd.read_csv(f"{BASE}/UDCF_RCT/core/hillstrom_predictions.txt", header=None)
df_cate.columns = ["cate_mens_email", "cate_womens_email"]

print("Dimensao dos CATEs:", df_cate.shape)
print("Numero de valores distintos por tratamento:")
print(df_cate.nunique())

test_ids = pd.read_csv(TEST_INDEX_PATH)
df_cate_com_id = pd.concat([test_ids.reset_index(drop=True), df_cate.reset_index(drop=True)], axis=1)
df_cate_com_id.to_csv("hillstrom_udcf_cate_replica_usuario.csv", index=False)

print("\n=== Resumo do CATE estimado (replica do pipeline do usuario) ===")
for nome, col in [("Mens E-Mail", "cate_mens_email"), ("Womens E-Mail", "cate_womens_email")]:
    c = df_cate[col]
    print(f"\n{nome}")
    print(f"  media: {c.mean():.4f} | desvio: {c.std():.4f} | min/max: {c.min():.4f} / {c.max():.4f}")

print("\nArquivo salvo: hillstrom_udcf_cate_replica_usuario.csv")